# Configuração do Ambiente

In [0]:
########################################################
#### LIBS
########################################################
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
#from pyspark.sql.functions import col
import pandas as pd
import numpy as np
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone

# Leitura dos dados

In [0]:
### função de leitura dos dados
def leitura_prata(dados):
     return spark.read.format("parquet").load(f"/Volumes/workspace/default/inep_avaliacao_alfabetizacao/silver/{dados}")

In [0]:
### leitura dos dados
alunos = leitura_prata("alunos")
municipio = leitura_prata("municipio")
uf = leitura_prata("uf")
meta_municipio = leitura_prata("meta_alfabetizacao_municipio")
meta_uf = leitura_prata("meta_alfabetizacao_uf")
meta_brasil = leitura_prata("meta_alfabetizacao_brasil")

ibge = leitura_prata("ibge_ideb")


# Construção da camada Ouro

## Município

In [0]:
municipio2 = (municipio
    .filter(F.col("ano") == 2023)
    .filter(F.col("rede") == "3")
    .select("id_municipio", "rede", "ano", "taxa_alfabetizacao", "media_portugues")
)

municipio2.show(5)
municipio2.count()

## Meta municipal

Note que a `taxa_alfabetização` presente na tabela das metas municipais é referente a 2024. No entanto, para a construção da camada ouro, e consequentemente para o ajuste dos, serão usados apenas os dados referentes ao anos de 2023.

Imputação dos dados

In [0]:
###
meta_municipio2 = (meta_municipio
    .filter(F.col("ano") == 2023)
    .select("id_municipio", "ano", "rede", "meta_alfabetizacao_2024", "meta_alfabetizacao_2030"))

meta_municipio2.show(5)
meta_municipio2.count()

## Dados de fontes externas

In [0]:
###
ibge2 = (ibge
    .filter(F.col("rede") == "Municipal")
    .withColumnsRenamed({
    "CD_MN": "id_municipio",
    "AREA_KM2": "area_km2",
    "POPULACAO": "pop"})  
)
ibge2.show(5)
ibge2.count()

# Rascunhos - Exemplos

In [0]:
"""
##
dados_final = (
    municipio2
    .groupBy("id_municipio")
    .count()
    .orderBy(F.col("count"))
    .withColumnRenamed("count", "n")
    .show()
)
##
x = municipio.select("id_municipio").distinct()
x.show()

###
remover_colunas = [c for c in alunos.columns if c.startswith("_")]

alunos2 = (alunos
    .filter(F.col("presenca") == 1)
    .drop(*remover_colunas)
    .drop("peso_aluno", "presenca", "preenchimento_caderno", "serie"))
"""
#(municipio2.count(), len(municipio2.columns))

#municipio2.filter(~F.col("id_municipio").isin(x)).show()

#dados_filtrado = dados.filter(~F.col("x").isin(v))

#meta_municipio.select("rede").distinct().show()

